In [27]:
import pandas as pd
import json

# Loading Data

In [28]:
input_path = "../data/interim/evaluations/gpt-4.1/test.csv"

In [29]:
df = pd.read_csv(input_path)
df

,df.index,jaccard_similarity,psj_similarity,is_executable,is_empty
0,0.0,1.000000,1.0,True,False
1,4.0,0.000000,1.0,True,False
2,11.0,0.333333,1.0,True,False
3,12.0,1.000000,1.0,True,False
4,13.0,1.000000,1.0,True,False
...,...,...,...,...,...
2466,4683.0,NaN,NaN,NaN,NaN
2467,4699.0,NaN,NaN,NaN,NaN
2468,4776.0,NaN,NaN,NaN,NaN
2469,4807.0,NaN,NaN,NaN,NaN


In [30]:
# create a mask for rows having NaN in jaccard_similarity or psj_similarity
mask = df["jaccard_similarity"].isna() | df["psj_similarity"].isna()
df[mask]

,df.index,jaccard_similarity,psj_similarity,is_executable,is_empty
2202,31.0,NaN,NaN,NaN,NaN
2203,80.0,NaN,NaN,NaN,NaN
2204,220.0,NaN,NaN,NaN,NaN
2205,323.0,NaN,NaN,NaN,NaN
2206,329.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...
2466,4683.0,NaN,NaN,NaN,NaN
2467,4699.0,NaN,NaN,NaN,NaN
2468,4776.0,NaN,NaN,NaN,NaN
2469,4807.0,NaN,NaN,NaN,NaN


In [31]:
df = df[~mask].copy()
df

,df.index,jaccard_similarity,psj_similarity,is_executable,is_empty
0,0.0,1.000000,1.00000,True,False
1,4.0,0.000000,1.00000,True,False
2,11.0,0.333333,1.00000,True,False
3,12.0,1.000000,1.00000,True,False
4,13.0,1.000000,1.00000,True,False
...,...,...,...,...,...
2197,4820.0,0.000000,0.99737,True,False
2198,4823.0,0.333333,1.00000,True,False
2199,4824.0,0.666667,1.00000,True,False
2200,4829.0,1.000000,1.00000,True,False


# Evaluation Metrics Statistics

In [32]:
df['exact_match'] = (df['jaccard_similarity'] == 1.0).apply(int)
df


,df.index,jaccard_similarity,psj_similarity,is_executable,is_empty,exact_match
0,0.0,1.000000,1.00000,True,False,1
1,4.0,0.000000,1.00000,True,False,0
2,11.0,0.333333,1.00000,True,False,0
3,12.0,1.000000,1.00000,True,False,1
4,13.0,1.000000,1.00000,True,False,1
...,...,...,...,...,...,...
2197,4820.0,0.000000,0.99737,True,False,0
2198,4823.0,0.333333,1.00000,True,False,0
2199,4824.0,0.666667,1.00000,True,False,0
2200,4829.0,1.000000,1.00000,True,False,1


In [33]:
print("Jaccard Similarity Statistics:")
print(df["jaccard_similarity"].describe())
print("\nPSJ Similarity Statistics:")
print(df["psj_similarity"].describe())
print("\nExact Match Statistics:")
print(df["exact_match"].describe())

Jaccard Similarity Statistics:
count    2202.000000
mean        0.462579
std         0.449883
min         0.000000
25%         0.000000
50%         0.333333
75%         1.000000
max         1.000000
Name: jaccard_similarity, dtype: float64

PSJ Similarity Statistics:
count    2202.000000
mean        0.778473
std         0.379973
min         0.000000
25%         0.666667
50%         1.000000
75%         1.000000
max         1.000000
Name: psj_similarity, dtype: float64

Exact Match Statistics:
count    2202.000000
mean        0.366031
std         0.481827
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000
Name: exact_match, dtype: float64


In [36]:
df_non_empty = df[(df["is_empty"] == False)]
print("Jaccard Similarity Statistics:")
print(df_non_empty["jaccard_similarity"].describe())
print("\nPSJ Similarity Statistics:")
print(df_non_empty["psj_similarity"].describe())
print("\nExact Match Statistics:")
print(df_non_empty["exact_match"].describe())

Jaccard Similarity Statistics:
count    2061.000000
mean        0.485007
std         0.447670
min         0.000000
25%         0.000000
50%         0.473168
75%         1.000000
max         1.000000
Name: jaccard_similarity, dtype: float64

PSJ Similarity Statistics:
count    2061.000000
mean        0.822027
std         0.341014
min         0.000000
25%         0.905983
50%         1.000000
75%         1.000000
max         1.000000
Name: psj_similarity, dtype: float64

Exact Match Statistics:
count    2061.000000
mean        0.381853
std         0.485959
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000
Name: exact_match, dtype: float64


# Trajectory Statistics

In [37]:
def get_stats_from_chat(messages: list[dict]) -> dict:
    stats = {
        "ai_messages": 0,
        "tool_calls": 0,
        "tool_errors": 0,
        "top_k_in_exec_cypher": 0,
    }
    for message in messages:
        msg_role = message["role"]
        if msg_role == "tool":
            stats["tool_calls"] += 1
            if "[ERROR]" in message["content"]:
                stats["tool_errors"] += 1
        elif msg_role == "assistant":
            stats["ai_messages"] += 1
            tool_calls = message.get("tool_calls", [])
            for call_str in tool_calls:
                call_dict = json.loads(call_str)
                fn = call_dict["function"]
                if fn['name'] == "execute_cypher_query":
                    first_k = fn['arguments'].get("first_k", None)

                    if first_k is not None:
                        stats["top_k_in_exec_cypher"] += 1
    return stats

In [38]:
# compute mean and std for ai_messages, tool_calls, tool_errors
stats = df["json_trajectory"].apply(lambda x: get_stats_from_chat(eval(x)))
# explode the stats dict into separate columns
stats_df = pd.json_normalize(stats)
stats_df

KeyError: 'json_trajectory'

In [ ]:
# for each column in stats_df, compute mean and std
for col in stats_df.columns:
    print(f"\nStatistics for {col}:")
    if col == 'top_k_in_exec_cypher':
        print(stats_df[col].sum())
    else:
        print(stats_df[col].describe())